In [2]:
import pandas as pd
from pathlib import Path
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import time
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Загружаем данные

Здесь файлы (столбцы в файлах разделены знаком препинания - запятой. CSV - значения разделенные запятой):
* train_data.csv - отвечает за обучение нейронной сети
* test.csv - тестовый датасет к которому нужно применить обученную нейронную сеть

## Переменные, которые они хранят

### Vmag, Bmag
* Vmag (Visual-Видимый) - яркость в желто-зеленом спектре
* Bmag (Blue-Синий) - яркость в синем спектре

Их разность = Bmag - Vmag дает нам показатель света. Если число маленькое или отрицательное - звезда горячая и голубая. Если большое - холодная и красная. (Рисунок 1 из методички)

### fuv_mag, nub_mag
* fuv_mag (Far Ultraviolet): Дальний ультрафиолет. Это свет с самой высокой энергией. Его излучают только экстремально горячие объекты (с температурой в десятки тысяч градусов).
* nuv_mag (Near Ultraviolet): Ближний ультрафиолет. Он находится на границе между тем, что видит человек (синий свет), и жестким излучением космоса.

### gpmag, rpmag, ipmag
* gpmag (green): Яркость в зеленом диапазоне
* rpmag (red): Яркость в красном диапазоне
* ipmag (infrared): Яркость в ближнем инфракрасном диапазоне. Она важна, потому что ИК-излучение лучше проходит сквозь космическую пыль

### e_***
Все переменные, начинающиеся на e_ (от слова error), — это стандартная ошибка или погрешность измерения соответствующей величины.

In [3]:
# Настройка путей
DATA_DIR = Path('data')
TRAIN_PATH = DATA_DIR / 'train_data.csv'
TEST_PATH = DATA_DIR / 'test.csv'
W4_PATH = DATA_DIR / 'w4_labeled_dataset.csv'

# Загружаем данные
# В train первая колонка — индекс, в test — колонка 'id'
train_original = pd.read_csv(TRAIN_PATH, index_col=0)
test_original = pd.read_csv(TEST_PATH, index_col='id')

print(f"Загружено: {len(train_original)} строк обучения и {len(test_original)} строк для теста.")

Загружено: 7000 строк обучения и 3000 строк для теста.


# Вычисление показателей света

## 1) Определение температуры (B-V)
Яркость звезды в отдельном фильтре (только Vmag или только Bmag) ничего не расскажет о ее природе, но разность между ними напрямую связна с температурой поверхности:
* Значение маленькое или отрицательное, зачит звезда горячая и голубая
* Значение большое - звезда холодная и красная

Большинство переменных звезд находятся строго в определенных значениях. Зная эти значения, нейронная сеть сразу понимает к какому температурному классу относится объект.
Также яркость меняется в зависимости от расстояния до "приемника", поэтому отдельные фильтры имеют огромный разброс точности, но их разность остается неизменной в зависимости от расстояния, что нам и надо

## 2) Поиск "аномальных" звезд через УФ-избыток (U-V)
Многие переменные звезды обладают активными внешними слоями (хромосферой) или являются очень горячими гигантами. Такие объекты «светятся» в ультрафиолете гораздо сильнее, чем обычные звезды.
Без даннного "маркера активности" нейронная сеть может перепутать переменную звезду с обычной, которая просто находится чуть дальше


In [4]:
def add_features(df):
    # Создаем копию, чтобы не изменять оригинал напрямуюпр
    df = df.copy()
    # Показатель цвета: Синий минус Видимый
    df['B-V'] = df['Bmag'] - df['Vmag']
    # Ультрафиолетовый избыток
    df['U-V'] = df['fuv_mag'] - df['Vmag']
    return df

train = add_features(train_original)
test = add_features(test_original)

print("Новые признаки успешно добавлены. Данные очищены (лишний шум выкинули, оставили только то, что пригодится в дальнейшем)")

Новые признаки успешно добавлены. Данные очищены (лишний шум выкинули, оставили только то, что пригодится в дальнейшем)


# Очистка, заполнение и нормализация

Делим датасет на перееменные X, Y, отвечающие за характеристику звезды и ответ - переменная она или нет соответственно
Заполнение пропусков происходит по среднему значению (нейтральному) всей колонки, которое сильно не повлияет на общую картину
Нормализация при помощи z-преобразования сводит значения к одному весу. Без него нейросеть может считать больший вес у одной характиристики, засчет того, что у нее больше число, а на деле другая с меньшим числом может иметь больший вес

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

feature_cols = [
    'Vmag', 'Bmag', 'gpmag', 'rpmag', 'ipmag', 
    'fuv_mag', 'nuv_mag', 'B-V', 'U-V', 
    'e_Vmag', 'e_Bmag'
]

X_train_raw = train[feature_cols]
y_train = train['variable']
X_test_raw = test[feature_cols]

# Заполняем пропуски "нейтральными элементами"
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train_raw) # запоминает медиану
X_test_imputed = imputer.transform(X_test_raw) # сразу меняет пустые значения на медиану, которую он запомнил на "тренировке"

# Нормализация и стандартизация
scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_imputed)
X_test_final = scaler.transform(X_test_imputed)

print(f"Форма обучающей выборки: {X_train_final.shape}")

Форма обучающей выборки: (7000, 11)


In [6]:
models = {
    "SVM": SVC(kernel='rbf', class_weight='balanced', random_state=777),
    "Random Forest": RandomForestClassifier(max_depth=10, n_estimators=500, class_weight='balanced', random_state=777),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=500, random_state=777)
}

results = []

print("Начинаю битву алгоритмов...\n")
acc_res = 0
acc_name = 'None'

# 2. Цикл обучения и оценки
for name, model in models.items():
    start_time = time.time()
    
    # Обучение
    model.fit(X_train_final, y_train)
    
    # Предсказание (на трейне для базовой проверки)
    y_pred = model.predict(X_train_final)
    
    end_time = time.time()
    
    # Считаем метрики
    acc = accuracy_score(y_train, y_pred)
    if acc > acc_res:
        acc_res = acc
        acc_name = name
    elapsed = end_time - start_time
    
    results.append({
        "Модель": name,
        "Accuracy": f"{acc:.4f}",
        "Время (сек)": f"{elapsed:.2f}"
    })
    
    print(f"✅ {name} завершен.")

# 3. Вывод сравнительной таблицы
print("\n--- ИТОГОВАЯ ТАБЛИЦА ---")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# 4. Подробный отчет для лучшей модели
print(f"\nДетальный отчет для {acc_name}:")
rf_pred = models[acc_name].predict(X_train_final)
print(classification_report(y_train, rf_pred))

Начинаю битву алгоритмов...

✅ SVM завершен.
✅ Random Forest завершен.
✅ Gradient Boosting завершен.

--- ИТОГОВАЯ ТАБЛИЦА ---
           Модель Accuracy Время (сек)
              SVM   0.7837        3.25
    Random Forest   0.9284        8.71
Gradient Boosting   0.9514       14.74

Детальный отчет для Gradient Boosting:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      6273
           1       0.98      0.54      0.70       727

    accuracy                           0.95      7000
   macro avg       0.97      0.77      0.84      7000
weighted avg       0.95      0.95      0.95      7000



## Анализ полученных результатов

Лучшим в точности показал себя градиентный бустинг (после изменений random_state, n_estimators, max_depth). Количество переменных звезд составило 727.

# Этап 3. Создание многоклассового классификатора

## Разделение на группы
Деление будет проходить на: Затменные системы (Eclipsing), RR Лиры (RR Lyrae), Цефеиды (Cepheids) и другие. Они имеют свои "коды типов" в переменной *vsc_type* в файле *w4_labeled_dataset.csv*.

In [8]:
df_labeled = pd.read_csv(W4_PATH)

def simplify_types_v2(vsx_type):
    vsx_type = str(vsx_type).upper()
    
    if any(x in vsx_type for x in ['EA', 'EB', 'EW']):
        return 'Eclipsing'
    elif 'RR' in vsx_type:
        return 'RR Lyrae'
    elif any(x in vsx_type for x in ['DCEP', 'CEP', 'CW']):
        return 'Cepheids'
    else:
        return 'Other'

# Применяем новую группировку
df_multi = df_labeled[df_labeled['target_variable'] == 1].copy()
df_multi['target_class'] = df_multi['vsx_type'].apply(simplify_types_v2)

print("Распределение по классам:")
print(df_multi['target_class'].value_counts())

Распределение по классам:
target_class
Other        25011
Eclipsing     2059
RR Lyrae      1005
Cepheids        52
Name: count, dtype: int64


## Интеграция

In [10]:
df_multi = add_features(df_multi)
X_multi_raw = df_multi[feature_cols]

# Кодируем названия классов в числа
le = LabelEncoder()
y_multi = le.fit_transform(df_multi['target_class'])

X_multi_final = scaler.transform(imputer.transform(X_multi_raw))

# 4. Создание и обучение нейросети (MLP)
mlp_multi = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32), 
    max_iter=1000, 
    random_state=777,
    early_stopping=True,
    validation_fraction=0.1
)

print("\nОбучаю многоклассовую нейросеть...")
mlp_multi.fit(X_multi_final, y_multi)

# 5. Оценка результатов
y_multi_pred = mlp_multi.predict(X_multi_final)

print("\n--- ОТЧЕТ ПО ТИПАМ ПЕРЕМЕННОСТИ ---")
print(classification_report(y_multi, y_multi_pred, target_names=le.classes_))


Обучаю многоклассовую нейросеть...

--- ОТЧЕТ ПО ТИПАМ ПЕРЕМЕННОСТИ ---
              precision    recall  f1-score   support

    Cepheids       0.00      0.00      0.00        52
   Eclipsing       0.70      0.31      0.43      2059
       Other       0.94      0.99      0.96     25011
    RR Lyrae       0.88      0.74      0.81      1005

    accuracy                           0.93     28127
   macro avg       0.63      0.51      0.55     28127
weighted avg       0.92      0.93      0.92     28127



/home/jupyter-vsm002/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jupyter-vsm002/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jupyter-vsm002/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

# Результаты

Получается очень плохая картина - критический дисбаланс разных классов. Цефеидов настолько мало в генеральной совокупности, что нейронная сеть использует "ленивый подход" и просто не рассматривает их.